# 📊 과제 1 - 전체 결과 통합 비교
각 Exp에서 저장한 체크포인트를 로드하여 한 번에 평가·비교

In [ ]:
# Cell 0: 패키지 설치
!pip install datasets wandb scikit-learn sentence-transformers gensim -q

In [ ]:
# Cell 1: 임포트
import torch, torch.nn as nn, torch.optim as optim
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import gensim.downloader as gensim_api
import numpy as np
import pandas as pd
import wandb

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 2: 데이터 로드
data = load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')

def remove_empty(row):
    return all(row[f] not in [None, ''] for f in ['id','text','label','sentiment'])

train_data = data['train'].filter(remove_empty)
dev_data   = data['validation'].filter(remove_empty)
test_data  = data['test'].filter(remove_empty)

output_size = len(set(train_data['label']))
test_labels = test_data['label']

print(f'Test set: {len(test_data)} samples')

In [ ]:
# Cell 3: MLP 클래스 정의
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, output_size)
        self.activation = nn.GELU()
        self.output_act = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(p=dropout_rate)
    
    def forward(self, x):
        x = self.dropout(self.activation(self.fc1(x)))
        x = self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

print('MLP 클래스 준비 완료')

In [ ]:
# Cell 4: 벡터화 - BoW (Exp1, Exp2)
print('BoW 벡터화 중...')
bow_vec = CountVectorizer()
bow_vec.fit(train_data['text'])
bow_test_v = bow_vec.transform(test_data['text'])
bow_test_t = torch.FloatTensor(bow_test_v.toarray()).to(device)
bow_input_size = bow_test_v.shape[1]
print(f'BoW input_size: {bow_input_size}')

In [ ]:
# Cell 5: 벡터화 - TF-IDF (Exp3)
print('TF-IDF 벡터화 중...')
tfidf_vec = TfidfVectorizer(max_features=30000)
tfidf_vec.fit(train_data['text'])
tfidf_test_v = tfidf_vec.transform(test_data['text'])
tfidf_test_t = torch.FloatTensor(tfidf_test_v.toarray()).to(device)
tfidf_input_size = tfidf_test_v.shape[1]
print(f'TF-IDF input_size: {tfidf_input_size}')

In [ ]:
# Cell 6: 벡터화 - GloVe (Exp4)
print('GloVe 로드 중...')
glove = gensim_api.load('glove-wiki-gigaword-100')

def texts_to_glove(texts, model, dim=100):
    vecs = []
    for sent in texts:
        words = sent.lower().split()
        wv = [model[w] for w in words if w in model]
        vecs.append(np.mean(wv, axis=0) if wv else np.zeros(dim))
    return np.array(vecs, dtype=np.float32)

glove_test_np = texts_to_glove(test_data['text'], glove)
glove_test_t = torch.FloatTensor(glove_test_np).to(device)
glove_input_size = 100
print(f'GloVe input_size: {glove_input_size}')

In [ ]:
# Cell 7: 벡터화 - MiniLM (Exp5)
print('MiniLM 로드 중...')
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
minilm_test_np = minilm.encode(test_data['text'], batch_size=128, 
                               show_progress_bar=True, convert_to_numpy=True)
minilm_test_t = torch.FloatTensor(minilm_test_np).to(device)
minilm_input_size = 384
print(f'MiniLM input_size: {minilm_input_size}')

In [ ]:
# Cell 8: 벡터화 - MPNet (Exp6)
print('MPNet 로드 중...')
mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
mpnet_test_np = mpnet.encode(test_data['text'], batch_size=64,
                             show_progress_bar=True, convert_to_numpy=True)
mpnet_test_t = torch.FloatTensor(mpnet_test_np).to(device)
mpnet_input_size = 768
print(f'MPNet input_size: {mpnet_input_size}')

In [ ]:
# Cell 9: 각 Exp 체크포인트 로드 및 평가
# ⚠️ Colab Files 패널에서 best_model_exp{1-6}.pt 업로드 필요

experiments = [
    # (name, checkpoint, input_size, hidden_size, dropout, test_tensor)
    ('Exp1 - BoW Baseline',      'best_model_exp1.pt', bow_input_size,   100,  0.0, bow_test_t),
    ('Exp2 - BoW + Sweep',       'best_model_exp2.pt', bow_input_size,   1000, 0.2, bow_test_t),
    ('Exp3 - TF-IDF + Sweep',    'best_model_exp3.pt', tfidf_input_size, 1000, 0.2, tfidf_test_t),
    ('Exp4 - GloVe + Sweep',     'best_model_exp4.pt', glove_input_size, 1000, 0.2, glove_test_t),
    ('Exp5 - MiniLM + Sweep',    'best_model_exp5.pt', minilm_input_size,1000, 0.2, minilm_test_t),
    ('Exp6 - MPNet + Sweep 🏆',  'best_model_exp6.pt', mpnet_input_size, 1000, 0.2, mpnet_test_t),
]

results = []

for name, ckpt_path, inp_size, hidden, dropout, test_t in experiments:
    try:
        # 모델 초기화
        model = MLP(inp_size, hidden, output_size, dropout).to(device)
        
        # 체크포인트 로드
        state_dict = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(state_dict)
        model.eval()
        
        # Test 평가
        with torch.no_grad():
            preds = torch.argmax(model(test_t), dim=1)
            acc = accuracy_score(test_labels, preds.cpu().tolist())
        
        results.append({
            'Experiment': name,
            'Test Accuracy': acc * 100,
            'Checkpoint': ckpt_path
        })
        print(f'✅ {name}: {acc*100:.2f}%')
        
    except FileNotFoundError:
        print(f'❌ {name}: {ckpt_path} 파일이 없습니다.')
        results.append({
            'Experiment': name,
            'Test Accuracy': 0.0,
            'Checkpoint': f'Missing: {ckpt_path}'
        })
    except Exception as e:
        print(f'❌ {name}: 오류 - {e}')
        results.append({
            'Experiment': name,
            'Test Accuracy': 0.0,
            'Checkpoint': f'Error: {e}'
        })

print('\n' + '='*70)

In [ ]:
# Cell 10: 결과 정리 표
df = pd.DataFrame(results)
df_sorted = df.sort_values('Test Accuracy', ascending=False)

print('\n📊 전체 실험 결과 (Test Accuracy 기준 내림차순)\n')
print(df_sorted.to_string(index=False))

# 최고 성능
best = df_sorted.iloc[0]
print('\n' + '='*70)
print(f"🏆 최고 성능: {best['Experiment']} — {best['Test Accuracy']:.2f}%")

# 베이스라인 대비 향상도
baseline_acc = df[df['Experiment'].str.contains('Baseline')]['Test Accuracy'].values[0]
improvement = best['Test Accuracy'] - baseline_acc
print(f"📈 베이스라인 대비 향상: +{improvement:.2f}%p")
print('='*70)

In [ ]:
# Cell 11: W&B에 결과 로깅 (선택)
# wandb.login()
# 
# wandb.init(project='nlp-hw1', name='results-summary', config={
#     'experiments': len(results),
#     'best_accuracy': best['Test Accuracy'],
#     'best_experiment': best['Experiment']
# })
# 
# # 각 실험 결과를 W&B Table로 로깅
# table = wandb.Table(dataframe=df_sorted)
# wandb.log({'results_table': table})
# 
# # 막대 그래프용 데이터
# for _, row in df_sorted.iterrows():
#     wandb.log({row['Experiment']: row['Test Accuracy']})
# 
# wandb.finish()
# print('✅ W&B에 결과 로깅 완료')